In [55]:
import pandas as pd
import string

path = "../../output/tnm/lubrication_switch.xlsx" 
xls = pd.ExcelFile(path)
print("Available sheets:", xls.sheet_names)


Available sheets: ['hitachi', 'srb', 'mhe-demag', 'mechanical-switch']


In [56]:
selected_sheets = xls.sheet_names

dfs = {s: xls.parse(s) for s in selected_sheets}

for name, df in dfs.items():
    if name == "Sheet":
        continue
    print(f"{name}: shape={df.shape}")
    # print(df.head())

hitachi: shape=(179, 102)
srb: shape=(44, 109)
mhe-demag: shape=(89, 42)
mechanical-switch: shape=(45, 109)


In [57]:
cols = []

for sheet in dfs:
    print(f"Processing sheet: {sheet}")
    cols.extend(dfs[sheet].columns.tolist())
    
    ## rename selected columns from those sheet into new name
    dfs[sheet].rename(columns={
        "technician.technician_id": "technician_id",
        "technician.date": "technician_date",
        "supervisor.supervisor_id": "supervisor_id",
        "supervisor.date": "supervisor_date",
    }, inplace=True)
    
cols

Processing sheet: hitachi
Processing sheet: srb
Processing sheet: mhe-demag
Processing sheet: mechanical-switch


['a.component',
 'a.lubricant',
 'a.amount',
 'a.interval',
 'a.completed?',
 'b.component',
 'b.lubricant',
 'b.amount',
 'b.interval',
 'b.completed?',
 'c.component',
 'c.lubricant',
 'c.amount',
 'c.interval',
 'c.completed?',
 'd.component',
 'd.lubricant',
 'd.amount',
 'd.interval',
 'd.completed?',
 'e.component',
 'e.lubricant',
 'e.amount',
 'e.interval',
 'e.completed?',
 'f.component',
 'f.lubricant',
 'f.amount',
 'f.interval',
 'f.completed?',
 'g.component',
 'g.lubricant',
 'g.amount',
 'g.interval',
 'g.completed?',
 'h.component',
 'h.lubricant',
 'h.amount',
 'h.interval',
 'h.completed?',
 'i.component',
 'i.lubricant',
 'i.amount',
 'i.interval',
 'i.completed?',
 'j.component',
 'j.lubricant',
 'j.amount',
 'j.interval',
 'j.completed?',
 'k.component',
 'k.lubricant',
 'k.amount',
 'k.interval',
 'k.completed?',
 'l.component',
 'l.lubricant',
 'l.amount',
 'l.interval',
 'l.completed?',
 'm.component',
 'm.lubricant',
 'm.amount',
 'm.interval',
 'm.completed?',

In [58]:
final_json = {}
lubswitch_reference = {}
bool_errors = []

In [59]:
GEARBOX_BRANDS = {"hitachi"}
DESCRIPTION_BRANDS = {"srb", "mechanical-switch"}

BRAND_RULES = {
    brand: {
        "use_description": brand in DESCRIPTION_BRANDS,
        "process_gearbox_oil": brand in GEARBOX_BRANDS,
    }
    for brand in GEARBOX_BRANDS | DESCRIPTION_BRANDS
}

DEFAULT_RULE = {"use_description": False, "process_gearbox_oil": False}

def build_status_key(col, row, use_description, component_counts):
    """
    Logic:
    1. Identify the component name (e.g., 'Pintle').
    2. Track how many times this component has appeared in this row.
    3. Assign a letter based on that count (a, b, c...).
    4. Reset happens naturally because component_counts is fresh for every row.
    """
    if not (".completed" in col):
        return None

    col_prefix = col.split('.completed')[0]
    comp_col = f"{col_prefix}.component"
    
    if comp_col not in row.index or pd.isna(row[comp_col]):
        return None

    raw_comp = str(row[comp_col]).strip().lower()
    comp_name = "".join(e if e.isalnum() or e == " " else "" for e in raw_comp).replace(" ", "_")

    current_idx = component_counts.get(comp_name, 0)
    dynamic_letter = string.ascii_lowercase[current_idx] # 0 -> 'a', 1 -> 'b'
    
    component_counts[comp_name] = current_idx + 1

    # if use_description:
    #     desc_col = f"{col_prefix}.description"
    #     desc_val = "status"
    #     if desc_col in row.index and pd.notna(row[desc_col]):
    #         desc_val = str(row[desc_col]).strip().lower().replace(" ", "_").replace("/", "_")
        
    #     return f"{comp_name}.{dynamic_letter}.status"
    
    return f"{comp_name}.{dynamic_letter}.status"

def get_inspection_item_col(completed_col):
    """
    Return the corresponding inspection_item column for a completed column
    """
    return completed_col.replace(".completed?", ".inspection_item").replace(".completed", ".inspection_item")

def to_bool(value, workorder_no=None, filename=None, column=None):
    """
    Convert various values to boolean for status columns.
    Unknown formats are stored as False and logged.
    """
    try:
        if pd.isna(value):
            return False

        if isinstance(value, bool):
            return value

        if isinstance(value, (int, float)):
            return value == 1

        if isinstance(value, str):
            v = value.strip().lower()
            if v in ["yes", "y", "true", "1", "checked", "ok", "✓"]:
                return True
            if v in ["no", "n", "false", "0", "unchecked", "x", "✗"]:
                return False

        # Unknown format
        if workorder_no and filename and column:
            bool_errors.append({
                "workorder_no": workorder_no,
                "filename": filename,
                "column": column,
                "value": value,
                "error": "Unknown boolean format, storing as False"
            })
        return False

    except Exception as e:
        if workorder_no and filename and column:
            bool_errors.append({
                "workorder_no": workorder_no,
                "filename": filename,
                "column": column,
                "value": value,
                "error": str(e)
            })
        return False
    
def process_lubswitch_sheet(df, brand, final_json):
    rule = BRAND_RULES.get(brand, DEFAULT_RULE)
    
    exclude_from_data = ["workorder_id", "filename"]
    metadata_cols = ["technician_id", "technician_name", "supervisor_id", "supervisor_name"]
    
    for _, row in df.iterrows():
        wo_id = str(row.get("workorder_id", ""))
        if not wo_id or wo_id == "nan": continue

        if wo_id not in final_json:
            final_json[wo_id] = {"filename": row.get("filename", "unknown"), "data": {}}

        component_counts = {}
        current_data = {}

        for col in df.columns:
            if col in exclude_from_data:
                continue

            if ".completed" in col:
                status_key = build_status_key(col, row, rule["use_description"], component_counts)
                if status_key:
                    current_data[status_key] = to_bool(row[col])
                continue

            if col in metadata_cols or (rule["process_gearbox_oil"] and "gearbox_oil_levels" in col):
                if pd.notna(row[col]):
                    current_data[col] = row[col]

        final_json[wo_id]["data"].update(current_data)

In [60]:
# 1. Initialize the storage containers
final_json = {}
lubswitch_reference = {}

# 2. Define the brands you want to process
target_brands = ["hitachi", "mhe-demag", "srb", "mechanical-switch"]

# 3. Loop through the brands and process if they exist in your 'dfs' dictionary
for brand in target_brands:
    df_brand = dfs.get(brand)
    
    if df_brand is not None and not df_brand.empty:
        print(f"Processing brand: {brand}...")
        process_lubswitch_sheet(
            df=df_brand, 
            brand=brand, 
            final_json=final_json
        )

# 4. (Optional) Convert to a standard JSON string for viewing
import json
print(json.dumps(final_json, indent=4))

Processing brand: hitachi...
Processing brand: mhe-demag...
Processing brand: srb...
Processing brand: mechanical-switch...
{
    "4000535202": {
        "filename": "TN_PM_MTH_LubricationSwitch2_4000535202.pdf",
        "data": {
            "swing_gearbox_upper_lip_seal_primary.a.status": true,
            "swing_gearbox_upper_lip_seal_secondary.a.status": true,
            "oil_level_primary_gearbox.a.status": true,
            "oil_level_secondary_gearbox.a.status": true,
            "refill_nonrumba_hand_grease_pump_reservoir.a.status": false,
            "oil_level_15kw_swing_motor_gearbox.a.status": true,
            "primary_to_secondary_gearbox_drive_shaft_couplings.a.status": true,
            "nonrumba_drive_mechanism_hand_pump.a.status": true,
            "nonrumba_external_mounting_pivot_points.a.status": true,
            "cam_guide_box_roller_path.a.status": true,
            "carriage_roller_axle_shaft.a.status": true,
            "locking_pin_assembly.a.status": true,


### Finalize JSON

In [61]:
import pandas as pd
import json

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/tnm/responses_tnm_lubswitch.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")


Saved as: ../../output/tnm/responses_tnm_lubswitch.xlsx
